# 04 - Forecasting

Trains LightGBM forecasting models on 2023 data and produces 366 daily predictions
per household for 2024.

**Three configurations:**
| Config | Description |
|---|---|
| `feature_ward` | One model per Feature+Ward cluster (cluster-level) |
| `kshape` | One model per k-Shape cluster (cluster-level) |
| `global` | Single model on all households, no clustering (dataset-level baseline) |

**Data leakage note:**
- `household_means` (used as a feature) are computed from 2023 train data only.
- Lag features for Jan 2024 look back into Dec 2023 - this uses 2023 *consumption* as context, not 2024 targets. Safe.
- 2024 values are never seen during model fitting.

Output CSVs are saved to `results/predictions/` for the evaluation teammate.

## Section 0 - Setup

In [24]:
import sys, os, warnings, pickle
warnings.filterwarnings('ignore')

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import lightgbm as lgb
from pathlib import Path

from src.utils.config import (
    RANDOM_SEED, LAG_FEATURES, ROLLING_WINDOWS,
    RESULTS_DIR, PREDICTIONS_DIR, DIAGNOSTICS_DIR,
)
from src.utils.data_loader import load_train_data, load_test_data
from src.forecasting.features       import prepare_features
from src.forecasting.cluster_loader import load_cluster_assignments
from src.forecasting.models         import (
    train_cluster_models, train_global_model, save_point_models,
)
from src.forecasting.predict import predict_point

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

# !! Update to match notebook 03 selected k values !!
CLUSTERING_CONFIG = {
    'feature_ward': {'k': 3},
    'kshape':       {'k': 2},
}

print(f'LightGBM        : {lgb.__version__}')
print(f'LAG_FEATURES    : {LAG_FEATURES}')
print(f'ROLLING_WINDOWS : {ROLLING_WINDOWS}')
print(f'RANDOM_SEED     : {RANDOM_SEED}')
print(f'Clustering k    : {CLUSTERING_CONFIG}')

LightGBM        : 4.6.0
LAG_FEATURES    : [1, 7, 14, 30]
ROLLING_WINDOWS : [7, 14, 30]
RANDOM_SEED     : 42
Clustering k    : {'feature_ward': {'k': 3}, 'kshape': {'k': 2}}


## Section 1 - Load Data

In [25]:
train_wide = load_train_data()   # 2023 - used for training only
test_wide  = load_test_data()    # 2024 - used for inference only

print(f'Train (2023): {train_wide.shape}  - households × days')
print(f'Test  (2024): {test_wide.shape}  - households × days')
assert set(train_wide.index) == set(test_wide.index), 'Household ID mismatch!'
print(f'Households: {len(train_wide):,}  ✓')

Train (2023): (17547, 365)  — households × days
Test  (2024): (17547, 366)  — households × days
Households: 17,547  ✓


## Section 2 - Shared Helper

Loads cluster assignments, builds features, attaches cluster labels.
Reused by all three configurations.

In [26]:
def setup_for_method(method, k, train_wide, test_wide):
    """Load clusters, build features, return everything needed to train."""
    cluster_series, degen_ids, reg_ids = load_cluster_assignments(
        DIAGNOSTICS_DIR, method, k
    )
    common         = cluster_series.index.intersection(train_wide.index)
    cluster_series = cluster_series.loc[common]
    reg_ids        = cluster_series[cluster_series >= 0].index
    degen_ids      = cluster_series[cluster_series <  0].index
    cluster_labels = sorted(cluster_series[cluster_series >= 0].unique().tolist())

    train_feats, test_feats, feature_cols, _ = prepare_features(
        train_wide, test_wide, reg_ids, LAG_FEATURES, ROLLING_WINDOWS
    )
    train_feats['cluster'] = train_feats['household_id'].map(cluster_series)
    test_feats['cluster']  = test_feats['household_id'].map(cluster_series)

    return cluster_series, reg_ids, degen_ids, cluster_labels, \
           train_feats, test_feats, feature_cols

print('Helper defined.')

Helper defined.


## Section 3 - Feature+Ward Cluster Models

One LightGBM model trained per cluster using only households in that cluster's 2023 data.
A global model is also trained as fallback for any unmapped households.

In [27]:
%%time
method_fw = 'feature_ward'
k_fw      = CLUSTERING_CONFIG[method_fw]['k']

cs_fw, reg_ids_fw, degen_ids_fw, labels_fw, \
train_feats_fw, test_feats_fw, feature_cols_fw = \
    setup_for_method(method_fw, k_fw, train_wide, test_wide)

print(f'Cluster labels : {labels_fw}')
print(f'Train features : {train_feats_fw.shape}')
print(f'Test  features : {test_feats_fw.shape}')
print(f'Feature cols   : {feature_cols_fw}')

  [feature_ward k=3] 17,547 regular, 0 degenerate households
Cluster labels : [0, 1, 2]
Train features : (5878245, 20)
Test  features : (6422202, 20)
Feature cols   : ['lag_1', 'lag_7', 'lag_14', 'lag_30', 'rolling_mean_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_std_14', 'rolling_mean_30', 'rolling_std_30', 'day_of_week', 'month', 'day_of_year', 'is_weekend', 'week_of_year', 'household_mean']
CPU times: user 2min 22s, sys: 18.7 s, total: 2min 41s
Wall time: 2min 43s


In [28]:
%%time
print('Training cluster models (2023 data only)...')
cluster_models_fw = train_cluster_models(
    train_feats_fw, feature_cols_fw, labels_fw, RANDOM_SEED
)

print('\nTraining global fallback model...')
global_model_fw = train_global_model(train_feats_fw, feature_cols_fw, RANDOM_SEED)

save_point_models(cluster_models_fw, global_model_fw,
                  RESULTS_DIR / 'models' / method_fw)
print('\n✓ Feature+Ward models trained and saved.')

Training cluster models (2023 data only)...
  [point] cluster 0: 1,067,980 train rows
  [point] cluster 1: 2,158,405 train rows
  [point] cluster 2: 2,651,860 train rows

Training global fallback model...
  [point] global model: 5,878,245 train rows
  Models saved to /home/hemali/rdkd/RDK Project (4)/results/models/feature_ward

✓ Feature+Ward models trained and saved.
CPU times: user 47min 8s, sys: 14 s, total: 47min 22s
Wall time: 14min 28s


In [29]:
%%time
test_degen_fw = test_wide.loc[degen_ids_fw.intersection(test_wide.index)]

preds_fw = predict_point(
    test_feats_fw, test_degen_fw, train_wide, feature_cols_fw,
    labels_fw, cluster_models_fw, global_model_fw, cs_fw, degen_ids_fw,
)

out_fw = PREDICTIONS_DIR / f'predictions_2024_{method_fw}.csv'
preds_fw.to_csv(out_fw, index=False)

days_per_hh = preds_fw.groupby('household_id').size()
print(f'Saved       : {out_fw}')
print(f'Households  : {preds_fw["household_id"].nunique():,}')
print(f'Days/hh     : {days_per_hh.min()}–{days_per_hh.max()}  (expected 366)')
print(f'\nSample output:')
display(preds_fw.head())

Saved       : /home/hemali/rdkd/RDK Project (4)/results/predictions/predictions_2024_feature_ward.csv
Households  : 17,547
Days/hh     : 366–366  (expected 366)

Sample output:


,household_id,date,predicted,cluster,model_used
0,22,2024-01-01,10.777168,1,cluster_lgbm
1,22,2024-01-02,9.824513,1,cluster_lgbm
2,22,2024-01-03,10.175415,1,cluster_lgbm
3,22,2024-01-04,10.521825,1,cluster_lgbm
4,22,2024-01-05,9.501039,1,cluster_lgbm


CPU times: user 14min 34s, sys: 2.64 s, total: 14min 37s
Wall time: 4min 45s


## Section 4 - k-Shape Cluster Models

Same pipeline, using k-Shape cluster assignments instead of Feature+Ward.

In [30]:
%%time
method_ks = 'kshape'
k_ks      = CLUSTERING_CONFIG[method_ks]['k']

cs_ks, reg_ids_ks, degen_ids_ks, labels_ks, \
train_feats_ks, test_feats_ks, feature_cols_ks = \
    setup_for_method(method_ks, k_ks, train_wide, test_wide)

print(f'Cluster labels : {labels_ks}')
print(f'Train features : {train_feats_ks.shape}')
print(f'Test  features : {test_feats_ks.shape}')

  [kshape k=2] 17,547 regular, 0 degenerate households
Cluster labels : [0, 1]
Train features : (5878245, 20)
Test  features : (6422202, 20)
CPU times: user 2min 21s, sys: 19.3 s, total: 2min 40s
Wall time: 2min 43s


In [31]:
%%time
print('Training k-Shape cluster models (2023 data only)...')
cluster_models_ks = train_cluster_models(
    train_feats_ks, feature_cols_ks, labels_ks, RANDOM_SEED
)

print('\nTraining global fallback model...')
global_model_ks = train_global_model(train_feats_ks, feature_cols_ks, RANDOM_SEED)

save_point_models(cluster_models_ks, global_model_ks,
                  RESULTS_DIR / 'models' / method_ks)
print('\n✓ k-Shape models trained and saved.')

Training k-Shape cluster models (2023 data only)...
  [point] cluster 0: 2,952,690 train rows
  [point] cluster 1: 2,925,555 train rows

Training global fallback model...
  [point] global model: 5,878,245 train rows
  Models saved to /home/hemali/rdkd/RDK Project (4)/results/models/kshape

✓ k-Shape models trained and saved.
CPU times: user 47min 19s, sys: 14.4 s, total: 47min 33s
Wall time: 14min 41s


In [32]:
%%time
test_degen_ks = test_wide.loc[degen_ids_ks.intersection(test_wide.index)]

preds_ks = predict_point(
    test_feats_ks, test_degen_ks, train_wide, feature_cols_ks,
    labels_ks, cluster_models_ks, global_model_ks, cs_ks, degen_ids_ks,
)

out_ks = PREDICTIONS_DIR / f'predictions_2024_{method_ks}.csv'
preds_ks.to_csv(out_ks, index=False)

days_per_hh = preds_ks.groupby('household_id').size()
print(f'Saved       : {out_ks}')
print(f'Households  : {preds_ks["household_id"].nunique():,}')
print(f'Days/hh     : {days_per_hh.min()}–{days_per_hh.max()}  (expected 366)')
display(preds_ks.head())

Saved       : /home/hemali/rdkd/RDK Project (4)/results/predictions/predictions_2024_kshape.csv
Households  : 17,547
Days/hh     : 366–366  (expected 366)


,household_id,date,predicted,cluster,model_used
0,22,2024-01-01,10.648435,0,cluster_lgbm
1,22,2024-01-02,10.268854,0,cluster_lgbm
2,22,2024-01-03,10.271526,0,cluster_lgbm
3,22,2024-01-04,10.805329,0,cluster_lgbm
4,22,2024-01-05,10.194218,0,cluster_lgbm


CPU times: user 14min 41s, sys: 1.94 s, total: 14min 43s
Wall time: 4min 38s


## Section 5 - Global Baseline (no clustering)

A single LightGBM model trained on all regular households together.
This is the dataset-level baseline required by the assignment.
Degenerate households still get their own 2023 historical mean.

In [33]:
%%time
# Use feature_ward cluster assignments only to identify degenerate households.
# The global model itself sees no cluster structure.
global_cs = cs_fw.copy()
global_cs.loc[reg_ids_fw] = 0    # all regular households -> single group

train_feats_gl, test_feats_gl, feature_cols_gl, _ = prepare_features(
    train_wide, test_wide, reg_ids_fw, LAG_FEATURES, ROLLING_WINDOWS
)
train_feats_gl['cluster'] = 0
test_feats_gl['cluster']  = 0

print(f'Train features : {train_feats_gl.shape}')
print(f'Test  features : {test_feats_gl.shape}')

Train features : (5878245, 20)
Test  features : (6422202, 20)
CPU times: user 2min 19s, sys: 16.2 s, total: 2min 35s
Wall time: 2min 37s


In [34]:
%%time
print('Training global model on all 2023 households...')
global_model_gl = train_global_model(train_feats_gl, feature_cols_gl, RANDOM_SEED)

save_point_models({0: global_model_gl}, global_model_gl,
                  RESULTS_DIR / 'models' / 'global')
print('\n✓ Global model trained and saved.')

Training global model on all 2023 households...
  [point] global model: 5,878,245 train rows
  Models saved to /home/hemali/rdkd/RDK Project (4)/results/models/global

✓ Global model trained and saved.
CPU times: user 22min 41s, sys: 12 s, total: 22min 53s
Wall time: 7min 20s


In [35]:
%%time
test_degen_gl = test_wide.loc[degen_ids_fw.intersection(test_wide.index)]

preds_gl = predict_point(
    test_feats_gl, test_degen_gl, train_wide, feature_cols_gl,
    [0], {0: global_model_gl}, global_model_gl, global_cs, degen_ids_fw,
)
preds_gl.loc[preds_gl['cluster'] == 0, 'model_used'] = 'global_lgbm'

out_gl = PREDICTIONS_DIR / 'predictions_2024_global.csv'
preds_gl.to_csv(out_gl, index=False)

days_per_hh = preds_gl.groupby('household_id').size()
print(f'Saved       : {out_gl}')
print(f'Households  : {preds_gl["household_id"].nunique():,}')
print(f'Days/hh     : {days_per_hh.min()}–{days_per_hh.max()}  (expected 366)')
display(preds_gl.head())

Saved       : /home/hemali/rdkd/RDK Project (4)/results/predictions/predictions_2024_global.csv
Households  : 17,547
Days/hh     : 366–366  (expected 366)


,household_id,date,predicted,cluster,model_used
0,22,2024-01-01,10.621974,0,global_lgbm
1,22,2024-01-02,10.401265,0,global_lgbm
2,22,2024-01-03,10.195419,0,global_lgbm
3,22,2024-01-04,10.882759,0,global_lgbm
4,22,2024-01-05,10.423908,0,global_lgbm


CPU times: user 15min 14s, sys: 1.46 s, total: 15min 15s
Wall time: 4min 44s


## Section 6 - Output Summary

In [36]:
print('=== FORECASTING COMPLETE ===')
print()
for label, path, preds in [
    ('feature_ward', out_fw, preds_fw),
    ('kshape',       out_ks, preds_ks),
    ('global',       out_gl, preds_gl),
]:
    n_hh   = preds['household_id'].nunique()
    n_days = preds.groupby('household_id').size()
    print(f'{label:<15}: {n_hh:,} households × {n_days.min()}–{n_days.max()} days')
    print(f'               -> {path}')
    print()

print('Columns in each CSV:', preds_fw.columns.tolist())
print()
print('Hand off to evaluation teammate:')
print(f'  {PREDICTIONS_DIR}')

=== FORECASTING COMPLETE ===

feature_ward   : 17,547 households × 366–366 days
               -> /home/hemali/rdkd/RDK Project (4)/results/predictions/predictions_2024_feature_ward.csv

kshape         : 17,547 households × 366–366 days
               -> /home/hemali/rdkd/RDK Project (4)/results/predictions/predictions_2024_kshape.csv

global         : 17,547 households × 366–366 days
               -> /home/hemali/rdkd/RDK Project (4)/results/predictions/predictions_2024_global.csv

Columns in each CSV: ['household_id', 'date', 'predicted', 'cluster', 'model_used']

Hand off to evaluation teammate:
  /home/hemali/rdkd/RDK Project (4)/results/predictions
